# SMS Phishing Dataset — Exploratory Data Analysis

This notebook explores the merged Sinhala/Singlish phishing dataset used for Milestone 2.

**Prerequisites:** Run `python scripts/merge_datasets.py` first if `dataset/processed/final_sms_phishing_dataset.csv` does not exist.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts._bootstrap import bootstrap

bootstrap()

from ml_engine.preprocessing.pipeline import preprocess_message

DATA_PATH = ROOT / "dataset/processed/final_sms_phishing_dataset.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run: python scripts/merge_datasets.py"
    )

df = pd.read_csv(DATA_PATH)
df["text_length"] = df["text"].astype(str).str.len()
print(f"Loaded {len(df):,} records")
df.head()

## 1. Class and source distribution

In [ ]:
label_counts = df["label"].value_counts()
source_counts = df["source"].value_counts()

print("Label distribution:")
print(label_counts)
print("\nSource distribution:")
print(source_counts)
print(f"\nPhishing rate: {label_counts.get('PHISHING', 0) / len(df):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

label_counts.plot(kind="bar", ax=axes[0], color=["#34A853", "#EA4335"])
axes[0].set_title("Label distribution")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")

source_counts.plot(kind="bar", ax=axes[1], color=["#4285F4", "#FBBC04"])
axes[1].set_title("Source distribution")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 2. Language breakdown (Google Form survey)

In [ ]:
survey = df[df["source"] == "google_form"].copy()
if not survey.empty:
    lang_counts = survey["language"].value_counts()
    print("Survey language tags:")
    display(lang_counts.to_frame("count"))

    print("\nOriginal survey categories:")
    display(survey["original_label"].value_counts().to_frame("count"))
else:
    print("No Google Form records found.")

## 3. Message length statistics

In [ ]:
length_stats = df.groupby("label")["text_length"].agg(["count", "mean", "min", "max"])
display(length_stats.round(1))

df.groupby(["source", "label"])["text_length"].mean().unstack(fill_value=0).round(1)

## 4. Sample messages by source and label

In [ ]:
for source in df["source"].unique():
    for label in ["PHISHING", "SAFE"]:
        subset = df[(df["source"] == source) & (df["label"] == label)]
        if subset.empty:
            continue
        sample = subset.iloc[0]
        print(f"--- {source} / {label} ---")
        print(sample["text"][:300])
        print()

## 5. Preprocessing pipeline preview

Shows raw vs cleaned text using `ml_engine.preprocessing.pipeline.preprocess_message`.

In [ ]:
preview = df[df["source"] == "google_form"].head(5)

for _, row in preview.iterrows():
    raw = str(row["text"])
    cleaned = preprocess_message(raw)
    print(f"ID: {row['text_id']} | {row['label']} | {row['language']}")
    print(f"RAW:     {raw[:200]}")
    print(f"CLEANED: {cleaned[:200]}")
    print("-" * 60)